<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/hands_on_ml_with_scikit-learn_Aurelien_textbook/Chap_15_Processing_Sequences_with_RNNs_and_CNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The outputs of a recurrent neuron at time step t is a function of all the inputs from previous time steps hence it can be said that RNNs have a form of memory. A part of a neural network that preserves some state across time steps is called a memory cell.
A single recurrent neuron, or a layer of recurrent neurons, is a very basic cell capable of learning only short patterns(typically about 10 time steps long).

--Input and Output Sequences --
1. Sequence-to-sequence network: An RNN can simultaneously take a sequence of inputs and produce a sequence of outputs. This type of network is useful for predicting time series such as stock prices.
2. Sequence-to-vector network: You could feed an RNN a sequence of inputs and ignore all of its outputs except for the last one. Eg, you could feed the network a sequence of words corresponding to a movie review and the network would output a sentiment score(eg. from -1[hate] to +1[love]
3. Vector-to-sequence network: You can feed the network the same input vector over and over again at each time step and let it output a sequence. Eg, the input could be an image(or the output of a CNN) and the output could be a caption for that image.
4. We can also have a sequence-to-vector network, called an encoder, followed by a vector-to-sequence network, called a decoder. Eg. This could be used for translating a sentence from one language to another. You would feed the network a sentence in one language, the encoder would convert this sentence into a single vector representation, and then the decoder would decode this vector into a sentence in another language. This two step model is called and Encoder-Decoder network and it works much better than trying to translate on the fly with a single sequence-to-sequence RNN as the last words of a sentence can affect the first words of the translation, so one needs to wait until it has seen the whole sentence before translating it.

### Training RNNs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from pathlib import Path
import tensorflow as tf

In [ ]:
%matplotlib
%matplotlib

Using matplotlib backend: <object object at 0x0000026AFFDF8950>
Using matplotlib backend: TkAgg


In [ ]:
# tf.keras.utils.get_file(
#     "ridership.tgz",
#     "https://github.com/ageron/data/raw/main/ridership.tgz",
#     cache_dir=".",
#     extract=True
# )

In [ ]:
path = Path("datasets/ridership/CTA_-_Ridership_-_Daily_Boarding_Totals.csv")
df = pd.read_csv(path, parse_dates=["service_date"])

In [ ]:
df.columns = ["date", "day_type", "bus", "rail", "total"]
df = df.sort_values('date').set_index("date")
df = df.drop("total", axis=1)
df = df.drop_duplicates()

In [ ]:
df["2019-03": "2019-05"].plot(grid=True, marker=".")

<Axes: xlabel='date'>

Naive forecasting: This involves copying the latest known value(eg. forecasting that tomorrow will be the same as today). However, this day, by viewing the data plot, shows strong signs of weekly seasonality so in this case, forecasting that next week will be the same as this week works better.

To visualize these naive forecasts, we'll overlay the two time series(bus and rail) as well as the same time series lagged by one week(i.e the time series is shifted towards the right). We'd also plot a difference between the time series and its lagged version(i.e the value at time t minus the value at time t-7). this is called Differencing

In [ ]:
diff_7 = df[["bus", "rail"]].diff(7)["2019-03": "2019-05"] # differencing
fig, axs = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
df.plot(ax=axs[0], legend=False, marker=".") # original time series
df.shift(7).plot(ax=axs[0], legend=False, linestyle=":") # lagged time series
diff_7.plot(ax=axs[1], marker=".") #7-day difference

<Axes: xlabel='date'>

From the plot above, we can see that the lagged time series closely tracks the original time series. when a time series is correlated with a lagged version of itself, we say that the time series is autocorrelated.

In [ ]:
diff_7.abs().mean() # mean absolute error (MAE)

bus     43915.608696
rail    42143.271739
dtype: float64

In [ ]:
targets = df[["bus", "rail"]]["2019-03": "2019-05"]
(diff_7 / targets).abs().mean() * 100 # mean absolute percentage error (MAPE)

bus     8.293847
rail    8.994765
dtype: float64

Looking at the time series, there doesn't appear to be any significant monthly seasonality, but lets check whether there's any yearly seasonality.

In [ ]:
period = slice("2001", "2019")
df_monthly = df.resample("M").mean(numeric_only=True) # computes the mean for each month
rolling_average_12_months = df_monthly[period].rolling(
    window=12).mean()
fig, ax = plt.subplots(figsize=(8, 4))
df_monthly[period].plot(ax=ax, marker=".")
rolling_average_12_months.plot(ax=ax, legend=False)

<Axes: xlabel='date'>

There seems to be some yearly seasonality in the data as well and it's more visible on the rail series than on the bus series.
Lets check what we get if we plot the 12-month difference

In [ ]:
df_monthly.diff(12)[period].plot(marker=".", figsize=(8, 3))

<Axes: xlabel='date'>

Differencing is a common technique used to remove trend and seasonality from a time series: it's easier to study a stationary time series, meaning one whose statistical properties remain constant over time, without any seasonality or trends. Once you're able to make accurate forecasts on the differenced time series, it's easy to turn them into forecasts for the actual time series by just adding back the past values that were previously subtracted.

### The ARMA Model Family

The Autoregressive Moving Average(ARMA) model computes its forecasts using a simple weighted sum of lagged values, and corrects these forecasts by adding a moving average. Specifically, the moving average component is computed using a weighted sum of the last few forecast errors. This model assumes that the time series is stationary hence differencing may help when using it.

Using differencing over a single time step will produce an approximation of the derivative of the time series, It will eliminate any linear trend, transforming the time step into a constant value. Eg if you apply one-step differencing to the series [3, 5, 7, 9, 11], you get the differenced time series [2, 2, 2, 2]. If the original time series has a quadratic trend instead of a linear trend, then a single round of differencing will not be enough. Eg, the series [1, 4, 9, 16, 25, 36] becomes [3, 5, 7, 9, 11] after one round of differencing, but if you run differencing for a second round, then you get [2, 2, 2, 2]. Running two rounds of differencing will eliminate quadratic trends. More generally, running d consecutive rounds of differencing computes an approximation of the dth order derivative of the time series, so it will eliminate polynomial trends up to degree d. d is called the order of integration.

ARiMA => Autoregressive Integrated Moving Average: This model runs d rounds of differencing to make the time series more stationary, then it applies a regular ARMA model. When making forecasts, it uses this ARMA model, then it adds back the terms that were subtracted by differencing.

SARIMA => Seasonal ARIMA: This models the time series in the same way as ARIMA but it additionally models a seasonal component for a given frequency(eg weekly) using the exact same ARIMA approach.

Fitting a SARIMA  model to the rail time series and using it to make a forecast for tomorrow's ridership

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
origin, today = "2019-01-01", "2019-05-31"
rail_series = df.loc[origin:today]["rail"].asfreq("D")
model = ARIMA(rail_series, order=(1, 0, 0),
              seasonal_order=(0, 1, 1, 7))
model = model.fit()
y_pred = model.forecast()
y_pred

2019-06-01    427758.626222
Freq: D, dtype: float64

in the code above, we use asfreq("D") to set the time series frequency to daily, this doesn't change the data since it is already daily, but without this, the ARIMA class would have to guess the frequency and it would display a warning.
Hyperparameters:
p => This determines how far back into the past the model should look when learning weights.
q => This determines how far back into the past the model should look when summing forecast errors using the learned weights.
d => The order of integration.
s =? This is the period of seasonal pattern.
p,q,d are hyperparameters of ARIMA while P,D,Q are hyperparameters that are used to model the seasonal pattern of SARIMA models.

order=(1, 0, 0) means p=1, d=0, q=0
seasonal_order=(0, 1, 1, 7) means P=0, D=1, Q=1, s=7

Lets run the code in a loop to make forecasts for everyday in March, April, and May and compute the MAE over that period.

In [ ]:
origin, start_date, end_date = "2019-01-01", "2019-03-01", "2019-05-31"
time_period = pd.date_range(start_date, end_date)
rail_series = df.loc[origin: end_date]["rail"].asfreq("D")
y_preds = []
for today in time_period.shift(-1):
    model = ARIMA(rail_series[origin:today],
                 order=(1, 0, 0),
                  seasonal_order=(0, 1, 1, 7))
    model = model.fit() # we retrain the model everyday
    y_pred = model.forecast()[0]
    y_preds.append(y_pred)
y_preds = pd.Series(y_preds, index=time_period)
mae = (y_preds - rail_series[time_period]).abs().mean()

In [ ]:
mae

32040.720090488467

The arima mae is much better as it is significantly lower than the MAE we got with naive forecasting(42143).

In [ ]:
pd.DataFrame(np.c_[y_preds, rail_series[time_period]], columns=["pred", "actual"]).plot()

<Axes: >

### Preparing the data for machine learning models

Let's try to use some machine learning models to forecast this time series.

Our Goal: To forecast tomorrow's ridership based on the ridership of the past 8 weeks of data (56 days).

Therefore, the inputs to our model will be sequences(Usually a single sequence per day once the model is in production), each containing 56 vales from time stes t - 55 to t(since we are considering the past 8 weeks). For each input sequence, the model will output a single value: the forecast for time step t + 1(i.e tomorrow)

What will we use as training data?
We will use every 56-day window from the past as training data and the target for each window will be the value immediately following it(it's next day ridership)

In [ ]:
import tensorflow as tf

The below code takes a time series containing the numbers 0 to 5 as input, and it builds a tf.data dataset containing all the windows of the desired length (3), as well as their corresponding targets, grouped into batches of size 2

In [ ]:
my_series = [0, 1, 2, 3, 4, 5]
my_dataset = tf.keras.utils.timeseries_dataset_from_array(
    my_series,
    targets=my_series[3:], # the targets are 3 steps into the future
    sequence_length=3,
    batch_size=2
)

In [ ]:
list(my_dataset)

[(<tf.Tensor: shape=(2, 3), dtype=int32, numpy=
  array([[0, 1, 2],
         [1, 2, 3]])>,
  <tf.Tensor: shape=(2,), dtype=int32, numpy=array([3, 4])>),
 (<tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[2, 3, 4]])>,
  <tf.Tensor: shape=(1,), dtype=int32, numpy=array([5])>)]

In [ ]:
# Using window() method of tf.data.Dataset to achieve the same result
for window_dataset in tf.data.Dataset.range(6).window(4, shift=1, drop_remainder=False):
    for element in window_dataset:
        print(f"{element}", end=" ")
    print()

# get rid of smaller windows by passing drop_remainder=True to the window method

0 1 2 3 
1 2 3 4 
2 3 4 5 
3 4 5 
4 5 
5 


The window() method returns a nested dataset, analogous to a list of lists, this is useful when you want to transform each window by calling its dataset methods(eg. to shuffle or batch them). However, we cannot use a nested_dataset directly for training as our model expects tensors as inputs, not datasets.
As such, we will call the flat_map() method to convert the nested dataset into a flat dataset(one that contains tensors, not datasets). The flat_map() method takes a function as an argument, which allows you to transform each dataset in the nested dataset before flattening. Eg, when you pass the fxn lambda ds: ds.batch(2) to flat_map(), then it will transform the nested dataset, say {{1, 2}, {3, 4, 5, 6}}, into the flat dataset {[1, 2], [3, 4], [5, 6]}, a dataset containing 3 tensors, each of size 2

In [ ]:
def to_windows(dataset, length):
    dataset = dataset.window(length, shift=1, drop_remainder=True)
    return dataset.flat_map(lambda window_dataset: window_dataset.batch(length))

In [ ]:
dataset = to_windows(tf.data.Dataset.range(6), 4) # 3 inputs + 1 target = 4
for window_tensor in dataset:
    print(f"{window_tensor}" )

[0 1 2 3]
[1 2 3 4]
[2 3 4 5]


In [ ]:
# split each window into inputs and targets
dataset = dataset.map(lambda window: (window [:-1], window[-1]))

In [ ]:
# group the resulting windows into batches of size 2
list(dataset.batch(2))

[(<tf.Tensor: shape=(2, 3), dtype=int64, numpy=
  array([[0, 1, 2],
         [1, 2, 3]], dtype=int64)>,
  <tf.Tensor: shape=(2,), dtype=int64, numpy=array([3, 4], dtype=int64)>),
 (<tf.Tensor: shape=(1, 3), dtype=int64, numpy=array([[2, 3, 4]], dtype=int64)>,
  <tf.Tensor: shape=(1,), dtype=int64, numpy=array([5], dtype=int64)>)]

Before we start training, we need to split the data into a training period, a validation period and a test period. We'll also scall it down by a factor of 1million(max value of the ridership is about 1.15million) to ensure the values are near the 0-1 range: this works nicely with the default weight initialization and learning rate

In [ ]:
# Lets focus on rail ridership for now
rail_train = df["rail"]["2016-01" : "2018-12"] / 1e6
rail_valid = df["rail"]["2019-01" : "2019-05"] / 1e6
rail_test = df["rail"]["2019-06":] / 1e6

When dealing with time series, one generally wants to split across time. However, in some cases, you may be able to split along other dimensions which will give you a longer time period to train on. For example, if you have data about financial health of 10,000 companies from 2001 to 2019, you might be able to split this data across the different companies.It's very likely that many of these companies will be strongly correlated though(eg whole economic sectors may go up or down jointly) and if you have correlated companies across the training set and the test set, your test set will not be as useful as its measure of the generalization error will be optimistically biased.

In [ ]:
seq_length = 56
train_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_train.to_numpy(),
    targets=rail_train[seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
valid_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_valid.to_numpy(),
    targets=rail_valid[seq_length:],
    sequence_length=seq_length,
    batch_size=32
)

### Forecasting Using a Linear Model

In [ ]:
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=[seq_length])
])
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_mae" , patience=50, restore_best_weights=True
)
opt = tf.keras.optimizers.SGD(learning_rate=0.02, momentum=0.9)
model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
history = model.fit(train_ds, validation_data=valid_ds, epochs=500, callbacks=[early_stopping_cb])

Epoch 1/500
33/33 [==============================] - 3s 9ms/step - loss: 0.0352 - mae: 0.1969 - val_loss: 0.0061 - val_mae: 0.0819
Epoch 2/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0080 - mae: 0.0938 - val_loss: 0.0044 - val_mae: 0.0675
Epoch 3/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0066 - mae: 0.0831 - val_loss: 0.0041 - val_mae: 0.0659
Epoch 4/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0063 - mae: 0.0811 - val_loss: 0.0044 - val_mae: 0.0696
Epoch 5/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0072 - mae: 0.0894 - val_loss: 0.0041 - val_mae: 0.0642
Epoch 6/500
33/33 [==============================] - 0s 7ms/step - loss: 0.0054 - mae: 0.0751 - val_loss: 0.0036 - val_mae: 0.0585
Epoch 7/500
33/33 [==============================] - 0s 6ms/step - loss: 0.0055 - mae: 0.0751 - val_loss: 0.0034 - val_mae: 0.0567
Epoch 8/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0051 - m

Epoch 125/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0026 - mae: 0.0439 - val_loss: 0.0026 - val_mae: 0.0438
Epoch 126/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0029 - mae: 0.0479 - val_loss: 0.0025 - val_mae: 0.0414
Epoch 127/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0025 - mae: 0.0428 - val_loss: 0.0022 - val_mae: 0.0387
Epoch 128/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0026 - mae: 0.0448 - val_loss: 0.0026 - val_mae: 0.0437
Epoch 129/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0028 - mae: 0.0471 - val_loss: 0.0031 - val_mae: 0.0515
Epoch 130/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0029 - mae: 0.0493 - val_loss: 0.0022 - val_mae: 0.0387
Epoch 131/500
33/33 [==============================] - 0s 5ms/step - loss: 0.0027 - mae: 0.0466 - val_loss: 0.0022 - val_mae: 0.0380
Epoch 132/500
33/33 [==============================] - 0s 5ms/step - 

In [ ]:
x = history.history

In [ ]:
pd.DataFrame(x)[["loss", "val_loss"]].plot()

<Axes: >